# Jano on BTS Airline On-Time Performance

This notebook uses a real extract from the U.S. Bureau of Transportation Statistics (BTS) Airline On-Time Performance data to show how [Jano](https://marmurar.github.io/jano/) works on an operational, time-dependent dataset.

Official source:

- BTS On-Time Performance overview: https://www.transtats.bts.gov/ontime/
- BTS table download page: https://www.transtats.bts.gov/DL_SelectFields.aspx?QO_fu146_anzr=b0-gvzr&gnoyr_VQ=FGJ
- Direct January 2024 PREZIP file used here: https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip

The dataset contains flight-level records reported by U.S. carriers, including scheduled departure time, actual departure time, actual arrival time, origin/destination airports, delay indicators and cancellation/diversion flags. That makes it useful for Jano because it has multiple operational timestamps and a realistic leakage problem: a flight can be scheduled or departed before its final arrival outcome is known.

This example uses the January 2024 monthly zip extract and filters it to flights departing from ATL so the notebook stays fast while still using real data.

It focuses on:

- loading a real monthly airline dataset from a local zip file,
- building leakage-aware temporal semantics with [`TemporalSemanticsSpec`](https://marmurar.github.io/jano/simulation.html#temporal-semantics-and-leakage-control),
- inspecting a walk-forward [`plan()`](https://marmurar.github.io/jano/concepts.html#planning-before-materialization) before materializing folds, including calendar-aligned daily windows,
- excluding special dates from the plan,
- and running temporal hypothesis policies on the same data.

Expected local file:

- `data/bts/bts_ontime_2024_01.zip`

Important: the BTS zip/CSV is intentionally kept out of git because it is a large external dataset. The repository ignores `data/`, so the notebook is versioned but the downloaded dataset is not.


In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import zipfile

import numpy as np
import pandas as pd

from jano import (
    DriftMonitoringPolicy,
    TemporalPartitionSpec,
    TemporalSemanticsSpec,
    TrainHistoryPolicy,
    WalkForwardPolicy,
)

BTS_ZIP_URL = "https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.zip"
DATA_PATH = Path("../data/bts/bts_ontime_2024_01.zip")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/bts/bts_ontime_2024_01.zip")


def ensure_bts_zip(path: Path = DATA_PATH, url: str = BTS_ZIP_URL) -> Path:
    """Download the BTS monthly zip only when it is missing locally."""
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        print(f"Downloading BTS sample to {path}...")
        urlretrieve(url, path)
    return path


DATA_PATH = ensure_bts_zip()
DATA_PATH


PosixPath('data/bts/bts_ontime_2024_01.zip')

## Load and prepare a workable sample

The monthly BTS file is large enough to demonstrate realistic fold planning. For notebook speed, we keep only flights departing from ATL and a compact set of columns.

In [2]:
def _hhmm_to_timedelta(value):
    if pd.isna(value):
        return pd.NaT
    value = int(value)
    hours, minutes = divmod(value, 100)
    return pd.Timedelta(hours=hours, minutes=minutes)


def _combine_flight_datetime(date_series, hhmm_series):
    base = pd.to_datetime(date_series)
    deltas = hhmm_series.map(_hhmm_to_timedelta)
    values = []
    for day, delta in zip(base, deltas):
        if pd.isna(delta):
            values.append(pd.NaT)
        else:
            values.append(day + delta)
    return pd.to_datetime(values)


def load_bts_sample(path: Path, origin: str = "ATL", max_rows: int = 25000) -> pd.DataFrame:
    with zipfile.ZipFile(path) as zf:
        csv_name = next(name for name in zf.namelist() if name.endswith(".csv"))
        columns = [
            "FlightDate",
            "Reporting_Airline",
            "Origin",
            "Dest",
            "DayOfWeek",
            "Distance",
            "Cancelled",
            "Diverted",
            "CRSDepTime",
            "DepTime",
            "CRSArrTime",
            "ArrTime",
            "ArrDel15",
            "DepDelayMinutes",
            "ArrDelayMinutes",
        ]
        frame = pd.read_csv(zf.open(csv_name), usecols=columns)

    frame = frame.loc[(frame["Origin"] == origin) & (frame["Cancelled"] == 0) & (frame["Diverted"] == 0)].copy()
    frame["scheduled_departure_at"] = _combine_flight_datetime(frame["FlightDate"], frame["CRSDepTime"])
    frame["actual_departure_at"] = _combine_flight_datetime(frame["FlightDate"], frame["DepTime"])
    frame["actual_arrival_at"] = _combine_flight_datetime(frame["FlightDate"], frame["ArrTime"])
    rollover = frame["actual_arrival_at"] < frame["actual_departure_at"]
    frame.loc[rollover, "actual_arrival_at"] = frame.loc[rollover, "actual_arrival_at"] + pd.Timedelta(days=1)
    frame["scheduled_dep_hour"] = (frame["CRSDepTime"].fillna(0).astype(int) // 100).astype(int)
    frame["is_arrival_delayed"] = frame["ArrDel15"].fillna(0).astype(int)
    frame = frame.dropna(subset=["scheduled_departure_at", "actual_arrival_at", "Distance", "DayOfWeek"])
    frame = frame.sort_values("scheduled_departure_at").head(max_rows).reset_index(drop=True)
    return frame


flights = load_bts_sample(DATA_PATH)
flights[["scheduled_departure_at", "actual_arrival_at", "Origin", "Dest", "Reporting_Airline", "is_arrival_delayed"]].head()

,scheduled_departure_at,actual_arrival_at,Origin,Dest,Reporting_Airline,is_arrival_delayed
0,2024-01-01 05:21:00,2024-01-01 07:22:00,ATL,FLL,NK,0
1,2024-01-01 05:35:00,2024-01-01 07:36:00,ATL,LGA,NK,0
2,2024-01-01 05:36:00,2024-01-01 07:13:00,ATL,MIA,AA,0
3,2024-01-01 05:47:00,2024-01-01 07:34:00,ATL,DFW,AA,0
4,2024-01-01 05:50:00,2024-01-01 07:34:00,ATL,MCO,F9,0


## Leakage-aware temporal semantics

The operational timeline is the scheduled departure. But supervised training should only see flights whose arrival outcome is already known.

That makes this a good fit for `TemporalSemanticsSpec`:

- timeline for reporting: `scheduled_departure_at`
- train eligibility: `actual_arrival_at`
- test eligibility: `scheduled_departure_at`

In [3]:
semantics = TemporalSemanticsSpec(
    timeline_col="scheduled_departure_at",
    segment_time_cols={
        "train": "actual_arrival_at",
        "test": "scheduled_departure_at",
    },
)

walk_forward = WalkForwardPolicy(
    time_col=semantics,
    partition=TemporalPartitionSpec(
        layout="train_test",
        train_size="7D",
        test_size="1D",
        calendar_frequency="D",
    ),
    step="1D",
    strategy="rolling",
    max_folds=8,
)
walk_forward

## Inspect the simulation plan before materializing folds

In Jano, a [`plan()`](https://marmurar.github.io/jano/concepts.html#planning-before-materialization) is the precomputed geometry of a simulation. It tells you what the future folds *would* look like before Jano materializes any `train` or `test` slices.

That matters because you can inspect the temporal design before training anything:

- how many iterations the policy creates,
- the start/end boundaries of each segment,
- how many rows each `train` and `test` window contains,
- whether the configuration creates sparse or empty periods,
- and which iterations should be excluded before materialization.

This is the main difference between planning and execution: the plan is a table of temporal intent; materialization turns that intent into actual fold objects and row indices.

Here we set `calendar_frequency="D"`, so duration windows are aligned to complete calendar days. Jano treats segment boundaries as closed-open intervals: `[start, end)`. A `train_end` of `2024-01-08 00:00:00` means train includes rows before Jan 8, while test starts exactly at Jan 8.


In [4]:
plan = walk_forward.plan(flights, title="ATL January 2024 walk-forward plan")
plan_df = plan.to_frame()
plan_df.head(10)

,iteration,fold,simulation_start,simulation_end,is_partial,train_start,train_end,train_rows,test_start,test_end,test_rows
0,0,0,2024-01-01,2024-01-09,False,2024-01-01,2024-01-08,5720,2024-01-08,2024-01-09,829
1,1,1,2024-01-02,2024-01-10,False,2024-01-02,2024-01-09,5829,2024-01-09,2024-01-10,792
2,2,2,2024-01-03,2024-01-11,False,2024-01-03,2024-01-10,5754,2024-01-10,2024-01-11,817
3,3,3,2024-01-04,2024-01-12,False,2024-01-04,2024-01-11,5738,2024-01-11,2024-01-12,853
4,4,4,2024-01-05,2024-01-13,False,2024-01-05,2024-01-12,5773,2024-01-12,2024-01-13,827
5,5,5,2024-01-06,2024-01-14,False,2024-01-06,2024-01-13,5737,2024-01-13,2024-01-14,759
6,6,6,2024-01-07,2024-01-15,False,2024-01-07,2024-01-14,5725,2024-01-14,2024-01-15,780
7,7,7,2024-01-08,2024-01-16,False,2024-01-08,2024-01-15,5655,2024-01-15,2024-01-16,842


The plan frame already exposes iteration ids, boundaries and row counts. That makes it possible to reason about the simulation before slicing any fold objects.

For example, you can use this table to choose a specific iteration, start from the middle of the simulation, or remove folds that overlap events that would bias training.


In [5]:
plan_df[["iteration", "train_start", "train_end", "test_start", "test_end", "train_rows", "test_rows"]]

,iteration,train_start,train_end,test_start,test_end,train_rows,test_rows
0,0,2024-01-01,2024-01-08,2024-01-08,2024-01-09,5720,829
1,1,2024-01-02,2024-01-09,2024-01-09,2024-01-10,5829,792
2,2,2024-01-03,2024-01-10,2024-01-10,2024-01-11,5754,817
3,3,2024-01-04,2024-01-11,2024-01-11,2024-01-12,5738,853
4,4,2024-01-05,2024-01-12,2024-01-12,2024-01-13,5773,827
5,5,2024-01-06,2024-01-13,2024-01-13,2024-01-14,5737,759
6,6,2024-01-07,2024-01-14,2024-01-14,2024-01-15,5725,780
7,7,2024-01-08,2024-01-15,2024-01-15,2024-01-16,5655,842


## Exclude special dates from the plan

As an example, exclude folds whose train segment overlaps Martin Luther King Jr. Day.

In [6]:
filtered_plan = plan.exclude_windows(train=[("2024-01-15", "2024-01-16")])
filtered_plan.to_frame()[["iteration", "train_start", "train_end", "test_start", "test_end"]]

,iteration,train_start,train_end,test_start,test_end
0,0,2024-01-01,2024-01-08,2024-01-08,2024-01-09
1,1,2024-01-02,2024-01-09,2024-01-09,2024-01-10
2,2,2024-01-03,2024-01-10,2024-01-10,2024-01-11
3,3,2024-01-04,2024-01-11,2024-01-11,2024-01-12
4,4,2024-01-05,2024-01-12,2024-01-12,2024-01-13
5,5,2024-01-06,2024-01-13,2024-01-13,2024-01-14
6,6,2024-01-07,2024-01-14,2024-01-14,2024-01-15
7,7,2024-01-08,2024-01-15,2024-01-15,2024-01-16


## Materialize the filtered simulation

In [7]:
simulation_result = filtered_plan.materialize()
simulation_result.total_folds, simulation_result.to_frame().head()

(8,    fold simulation_start simulation_end  ... test_start   test_end  test_rows
0     0       2024-01-01     2024-01-09  ... 2024-01-08 2024-01-09        829
1     1       2024-01-02     2024-01-10  ... 2024-01-09 2024-01-10        792
2     2       2024-01-03     2024-01-11  ... 2024-01-10 2024-01-11        817
3     3       2024-01-04     2024-01-12  ... 2024-01-11 2024-01-12        853
4     4       2024-01-05     2024-01-13  ... 2024-01-12 2024-01-13        827

[5 rows x 9 columns])

## A minimal model for temporal hypothesis policies

To keep this notebook self-contained, use a simple historical delay-rate classifier keyed by route, airline, day of week and scheduled departure hour.

In [8]:
class SimpleDelayRateClassifier:
    def fit(self, X: pd.DataFrame, y: pd.Series):
        keys = self._keys(X)
        grouped = pd.DataFrame({"key": keys, "target": y.to_numpy()}).groupby("key")["target"].mean()
        self.rate_by_key = grouped.to_dict()
        self.global_rate = float(np.mean(y.to_numpy()))
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        keys = self._keys(X)
        return np.array([
            1 if self.rate_by_key.get(key, self.global_rate) >= 0.5 else 0
            for key in keys
        ])

    @staticmethod
    def _keys(X: pd.DataFrame) -> pd.Series:
        return (
            X[["Origin", "Dest", "Reporting_Airline", "DayOfWeek", "scheduled_dep_hour"]]
            .astype(str)
            .agg("|".join, axis=1)
        )


feature_cols = ["Origin", "Dest", "Reporting_Airline", "DayOfWeek", "scheduled_dep_hour", "Distance"]
model = SimpleDelayRateClassifier()

## Train history question

How much historical data is enough to classify delayed arrivals on the same test day?

In [9]:
train_history = TrainHistoryPolicy(
    semantics,
    cutoff=plan_df.iloc[3]["train_end"],
    train_sizes=["2D", "4D", "6D", "7D"],
    test_size="1D",
)

train_history_result = train_history.evaluate(
    flights,
    model=model,
    target_col="is_arrival_delayed",
    feature_cols=feature_cols,
    metrics="accuracy",
)

train_history_result.to_frame()

,variant,train_size,train_start,train_end,test_start,test_end,train_rows,test_rows,accuracy
0,0,2 days 00:00:00,2024-01-09,2024-01-11,2024-01-11,2024-01-12,1615,853,0.855803
1,1,4 days 00:00:00,2024-01-07,2024-01-11,2024-01-11,2024-01-12,3279,853,0.855803
2,2,6 days 00:00:00,2024-01-05,2024-01-11,2024-01-11,2024-01-12,4908,853,0.853458
3,3,7 days 00:00:00,2024-01-04,2024-01-11,2024-01-11,2024-01-12,5738,853,0.831184


In [10]:
train_history_result.find_optimal_train_size(metric="accuracy", tolerance=0.0, relative=False)

{'variant': 0, 'train_size': '2 days 00:00:00', 'train_start': Timestamp('2024-01-09 00:00:00'), 'train_end': Timestamp('2024-01-11 00:00:00'), 'test_start': Timestamp('2024-01-11 00:00:00'), 'test_end': Timestamp('2024-01-12 00:00:00'), 'train_rows': 1615, 'test_rows': 853, 'accuracy': 0.8558030480656507}

## Drift monitoring question

If we freeze the train window, when does accuracy begin to degrade as test moves forward?

In [11]:
drift_monitor = DriftMonitoringPolicy(
    semantics,
    cutoff=plan_df.iloc[3]["train_end"],
    train_size="7D",
    test_size="1D",
    step="1D",
    max_windows=6,
)

drift_result = drift_monitor.evaluate(
    flights,
    model=model,
    target_col="is_arrival_delayed",
    feature_cols=feature_cols,
    metrics="accuracy",
)

drift_result.to_frame()

,window,train_size,train_start,train_end,test_start,test_end,train_rows,test_rows,accuracy
0,0,7 days 00:00:00,2024-01-04,2024-01-11,2024-01-11,2024-01-12,5738,853,0.831184
1,1,7 days 00:00:00,2024-01-04,2024-01-11,2024-01-12,2024-01-13,5738,827,0.637243
2,2,7 days 00:00:00,2024-01-04,2024-01-11,2024-01-13,2024-01-14,5738,759,0.689065
3,3,7 days 00:00:00,2024-01-04,2024-01-11,2024-01-14,2024-01-15,5738,780,0.723077
4,4,7 days 00:00:00,2024-01-04,2024-01-11,2024-01-15,2024-01-16,5738,842,0.595012
5,5,7 days 00:00:00,2024-01-04,2024-01-11,2024-01-16,2024-01-17,5738,824,0.514563


In [12]:
drift_result.find_drift_onset(metric="accuracy", threshold=0.05, baseline="first", relative=False)

{'window': 1, 'train_size': '7 days 00:00:00', 'train_start': Timestamp('2024-01-04 00:00:00'), 'train_end': Timestamp('2024-01-11 00:00:00'), 'test_start': Timestamp('2024-01-12 00:00:00'), 'test_end': Timestamp('2024-01-13 00:00:00'), 'train_rows': 5738, 'test_rows': 827, 'accuracy': 0.6372430471584039}

## Notes

- This notebook is intentionally pragmatic: it shows how to use Jano on a real airline dataset rather than trying to build a production-grade flight delay model.
- The main Jano-specific point is the temporal geometry and leakage-aware semantics, not the classifier sophistication.
- For heavier experimentation, keep the same temporal setup and replace the toy classifier with your own pipeline.